# A Little Extra!  
## New Addition to Week 1 by Instructor
### The Unreasonable Effectiveness of the Agent Loop

## What is an Agent?  
### Three compeeting definitions  
1. AI systems that can do work for you independtenly - Sam Altman.
2. A system in which an LLM controls the workflow - Anthropic.
3. An LLM agent runs tolls in a loop to achieve a goal.  

### **The third one is the new, emerging definition**

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override = True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists
todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo # {index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index +1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todo(description: list[str]) -> str:
    todos.extend(description)
    completed.extend([False] * len(description))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_note: str) -> str:
    if 1 <= index <= len(todos):
        completed[index -1] = True
    else:
        return "No todo at this index"
    Console().print(completion_note)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todo(["Buy groceries", 'Finish extra lab', 'Eat banana'])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo # 1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo # 1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    'name' : 'create_todo',
    'description' : 'Add new todos from a list of descriptions and return the full list',
    'parameters' : {
        'type' : 'object',
        'properties' : {
            'description': {
                'type' : 'array',
                'items' : {'type': 'string'},
                'title' : 'Description'
            }
        },
        'required' : ['description'],
        'additionalProperties' : False
    }
}

In [12]:
mark_complete_json = {
    'name' : 'mark_complete',
    'description' : "Mark complete the todo at the given position (starting from 1) and return the full list",
    'parameters' : {
        'properties' : {
            'index' : {
                'description' : "The 1-based index of the todo to mark as complete",
                'title' : 'Index',
                'type' : 'integer',
                
                
            },
            'completion_note' : {
                'description' : 'Notes about how you completed the todo in rich console markup',
                'title' : 'Completion Notes',
                'type' : 'string'
            }
        },
        'required' : ['index', 'completion_note'],
        'type' : 'object',
        'additionalProperties' : False
    }
}

In [13]:
tools = [{'type': 'function', 'function' : create_todos_json},
         {'type': 'function', 'function': mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({'role': 'tool', 'content' : json.dumps(result), 'tool_call_id': tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model = 'gpt-4o-mini', messages = messages, tools = tools, reasoning_effort='none')
        finish_reason = response.choices[0].finish_reason
        if finish_reason == 'tool_calls':
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
        
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

BadRequestError: Error code: 400 - {'error': {'message': 'Unrecognized request argument supplied: reasoning_effort', 'type': 'invalid_request_error', 'param': None, 'code': None}}